# Phase Retrieval : PSF → WFE → Zernikes

Convertit une PSF (+ un masque pupillaire) en coefficients de Zernike
(ordre de Noll), via trois méthodes interchangeables derrière un point
d'entrée unique (`pipeline.psf_to_wfe`) :

| Méthode | `method=` | Entrées | Principe |
|---|---|---|---|
| Gerchberg-Saxton | `"gerchberg_saxton"` | 1 PSF | Récupération de phase itérative par projections FFT successives, puis décomposition Zernike de la phase obtenue |
| Optimisation directe | `"optimization"` | 1 PSF | Ajustement des coefficients de Zernike par moindres carrés non-linéaires (`scipy.optimize.least_squares`) sur la PSF |
| Diversité de phase modale | `"phase_diversity"` | **2 PSF** (focus + défocus connu) | Ajustement conjoint des coefficients sur les deux PSF, ce qui lève l'ambiguïté de signe des deux méthodes précédentes |

**Tout le code numérique est externalisé** dans le paquet
`eoqual.phaseretrieval` (voir la section Imports) : ce notebook ne fait
qu'appeler ce paquet et présenter/valider les résultats.

## Sommaire

| Section | Contenu | Détail théorique |
|---|---|---|
| Méthode 1/2/3 | Les trois méthodes sur les mêmes données simulateur | §4, §6.3 |
| Validation croisée | Comparaison RMS/corrélation des trois méthodes à la vérité terrain | §6.4 |
| Lever l'ambiguïté de signe | A priori sur un mode pair (Z13/Z15), robustesse GS vs. optimisation | §3.7 |
| Visualisation interactive | Diagnostics Plotly (`verbose=True`) | §9.4 |
| *Fonctionnalités avancées* : |  |  |
| — Module de simulation | PSF/paire focus-défocus en mémoire, sans fichier | §6.7 |
| — Pupille obstruée | `orthogonal_basis` + retour à la base standard | §10 |
| — WFE haut ordre connue | `known_ho_wfe_nm` | §11.3 |
| — A priori individuels | `zernike_priors` / `zernike_bounds` | §6.5bis |
| — Suréchantillonnage en mémoire | `aperture_px` | §12.5 |
| — Exemple : lire + décomposer une PSF | `imageio` + `pandas` + comparaison Plotly (2D, coupes H/V) | — |
| — Exemple : tirage aléatoire de Zernike | Budget de RMS, décroissance radiale réglable, RMSE de la WFE récupérée | §6.5bis |

L'interface en ligne de commande et l'organisation du code (arborescence
du paquet) sont documentées dans
[`README.md`](../README.md#interface-en-ligne-de-commande) plutôt qu'ici
— plus à leur place dans une référence stable que dans un notebook.

## Sur la fusion

Les méthodes Gerchberg-Saxton et optimisation partagent la même signature
(une seule PSF) : les fusionner derrière un simple sélecteur `method=` dans
[`pipeline.psf_to_wfe`](../src/eoqual/phaseretrieval/pipeline.py) est direct et évite la duplication de
tout le code commun (chargement PSF/masque, décomposition Zernike finale,
diagnostics).

La diversité de phase, elle, a besoin d'une **seconde** PSF et d'un défocus
connu : ce n'est pas une simple variante d'algorithme mais un protocole de
mesure différent. Elle **est** intégrée au même point d'entrée unifié
(`method="phase_diversity"`, avec `diversity_path=` et `defocus=`
supplémentaires) pour bénéficier des mêmes diagnostics et de la même
interface CLI, mais son cœur numérique reste dans son propre module
([`backends/phase_diversity.py`](../src/eoqual/phaseretrieval/backends/phase_diversity.py)) plutôt que
d'être mélangé aux méthodes mono-PSF.

In [1]:
if not '__file__' in globals():  #HACK: pour éviter ces lignes non commentées quand on utilise manuellement nbconvert
    %load_ext autoreload
    %autoreload 2

## Imports

In [2]:
# Rend le paquet importable sans installation préalable
# (pip install -e . depuis la racine du dépôt fonctionne aussi).
import sys
from pathlib import Path as _Path
sys.path.insert(0, str((_Path.cwd() / ".." / "src").resolve()))

# ======================================================================
# IMPORTS STANDARD / TIERS / MODULES DU PROJET
# ======================================================================
# IMPORTS STANDARD
from pathlib import Path

# IMPORTS TIERS
import numpy as np
import plotly.graph_objects as go
from loguru import logger

# MODULES DU PROJET (voir les fichiers .py à côté de ce notebook)
from eoqual.phaseretrieval.io_utils import load_image, load_and_prepare_mask
from eoqual.phaseretrieval.zernike_fit import ZernikeConfig, fit_zernike
from eoqual.phaseretrieval.pipeline import psf_to_wfe, METHODS

logger.success("Fin des imports !")

2026-11-01 15:20:07.774 | SUCCESS  | __main__:<module>:23 - Fin des imports !


## Données du simulateur & paramètres communs

On utilise les données produites par `simu_psf_mask.ipynb` (`Data/*_simu.tiff`) pour valider les trois méthodes : la phase injectée (`phase_simu.tiff`) sert de vérité terrain, décomposée elle aussi sur la base de Zernike pour servir de référence de comparaison.

In [3]:
DATA_DIR = Path("data")

# Paramètres physiques communs aux trois méthodes
COMMON = dict(
    psf_path=DATA_DIR / "psf_simu.tiff",
    mask_path=DATA_DIR / "mask_simu.tiff",
    wavelength=550e-9,
    focal_length=4.0,
    pupil_diameter=0.5,
    pixel_pitch=4e-6,
    zernike_order=15,
    verbose=False,  # mettre à True pour les diagnostics Plotly interactifs
)

# Vérité terrain : décomposition Zernike de la phase injectée par le simulateur
mask_ref, n_pup_target_ref = load_and_prepare_mask(
    psf_shape=(256, 256),
    mask_path=COMMON["mask_path"],
    wavelength=COMMON["wavelength"],
    focal_length=COMMON["focal_length"],
    pupil_diameter=COMMON["pupil_diameter"],
    pixel_pitch=COMMON["pixel_pitch"],
)
phase_true_rad = load_image(DATA_DIR / "phase_simu.tiff")
coeffs_true_nm, _, _ = fit_zernike(
    phase_true_rad, mask_ref, ZernikeConfig(order=COMMON["zernike_order"], wavelength=COMMON["wavelength"])
)

print("Coefficients de Zernike de la vérité terrain (nm WFE) :")
for i, c in enumerate(coeffs_true_nm, start=1):
    if abs(c) > 1e-6:
        print(f"   > Z{i:2d}: {c:8.2f} nm")

2026-11-01 15:20:07.815 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:476 - Masque sur la grille de la PSF (256x256) mais ouverture détectée plus petite que le cadre (205x205 px) : recadrée/recentrée sans changement de résolution (voir REFERENCE_TECHNIQUE.md §12.3/§12.5).


Coefficients de Zernike de la vérité terrain (nm WFE) :
   > Z 4:    35.20 nm
   > Z 5:    17.60 nm


## Méthode 1 — Gerchberg-Saxton (`method="gerchberg_saxton"`, 1 PSF)

Récupération de phase itérative par projections successives entre plan pupille et plan focal. L'initialisation de la phase pupillaire est **aléatoire** (`seed=` pour la reproductibilité) : l'algorithme converge parfois vers la phase correcte, parfois vers son opposé (ambiguïté de signe), et peut aussi stagner selon le tirage initial (voir la section Validation).

In [4]:
coeffs_gs, phase_fit_gs_nm, diag_gs = psf_to_wfe(
    method="gerchberg_saxton", max_iter=300, seed=2, **COMMON,
)

print("Coefficients Gerchberg-Saxton (nm WFE) :")
for i, c in enumerate(coeffs_gs[:8], start=1):
    print(f"   > Z{i:2d}: {c:8.2f} nm")
print("Diagnostics :", diag_gs)

2026-11-01 15:20:07.859 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:476 - Masque sur la grille de la PSF (256x256) mais ouverture détectée plus petite que le cadre (205x205 px) : recadrée/recentrée sans changement de résolution (voir REFERENCE_TECHNIQUE.md §12.3/§12.5).


Coefficients Gerchberg-Saxton (nm WFE) :
   > Z 1:     0.00 nm
   > Z 2:     0.00 nm
   > Z 3:     0.00 nm
   > Z 4:    24.73 nm
   > Z 5:    24.13 nm
   > Z 6:     0.98 nm
   > Z 7:    -0.67 nm
   > Z 8:    -0.64 nm
Diagnostics : {'method': 'gerchberg_saxton', 'matrix_rank': 15, 'condition_number': 1.0973200384207147, 'phase_rms_nm': 37.23624028145434, 'fit_rms_nm': 35.815398844742184, 'residual_rms_nm': 113.12041499603373, 'residual_std_nm': 10.33405282383154}


## Méthode 2 — Optimisation directe (`method="optimization"`, 1 PSF)

Les coefficients de Zernike sont ajustés directement pour que la PSF simulée reproduise la PSF mesurée (`scipy.optimize.least_squares`). Plus rapide et déterministe que Gerchberg-Saxton (pas d'initialisation aléatoire), mais **souffre de la même ambiguïté de signe** : rien dans une PSF unique ne permet de distinguer une aberration de son opposée.

In [5]:
coeffs_opt, phase_fit_opt_nm, diag_opt = psf_to_wfe(
    method="optimization", fit_piston_tilt=False, **COMMON,
)

print("Coefficients Optimisation directe (nm WFE) :")
for i, c in enumerate(coeffs_opt[:8], start=1):
    print(f"   > Z{i:2d}: {c:8.2f} nm")
print("Diagnostics :", diag_opt)

2026-11-01 15:20:08.523 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:476 - Masque sur la grille de la PSF (256x256) mais ouverture détectée plus petite que le cadre (205x205 px) : recadrée/recentrée sans changement de résolution (voir REFERENCE_TECHNIQUE.md §12.3/§12.5).


Coefficients Optimisation directe (nm WFE) :
   > Z 1:     0.00 nm
   > Z 2:     0.00 nm
   > Z 3:     0.00 nm
   > Z 4:    26.65 nm
   > Z 5:    15.92 nm
   > Z 6:     0.00 nm
   > Z 7:    -0.88 nm
   > Z 8:    -0.88 nm
Diagnostics : {'method': 'optimization', 'cost': 0.002495413568275719, 'nfev': 21, 'status': 2, 'message': '`ftol` termination condition is satisfied.', 'psf_rmse': 9.28789295529179e-05}


## Méthode 3 — Diversité de phase modale (`method="phase_diversity"`, 2 PSF)

Une seconde PSF, acquise avec un défocus connu (`defocus=`), est ajustée **conjointement** avec la PSF focalisée. C'est la seule des trois méthodes qui nécessite `diversity_path=` — l'information supplémentaire apportée par le second cliché lève l'ambiguïté de signe des deux méthodes précédentes.

In [6]:
coeffs_pd, phase_fit_pd_nm, diag_pd = psf_to_wfe(
    method="phase_diversity",
    diversity_path=DATA_DIR / "psf_diversity_simu.tiff",
    defocus=275e-9,
    roi_radius=32,
    **COMMON,
)

print("Coefficients Diversité de phase (nm WFE) :")
for i, c in enumerate(coeffs_pd[:8], start=1):
    print(f"   > Z{i:2d}: {c:8.2f} nm")
print("Diagnostics :", diag_pd)

2026-11-01 15:20:09.391 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:476 - Masque sur la grille de la PSF (256x256) mais ouverture détectée plus petite que le cadre (205x205 px) : recadrée/recentrée sans changement de résolution (voir REFERENCE_TECHNIQUE.md §12.3/§12.5).


Coefficients Diversité de phase (nm WFE) :
   > Z 1:     0.00 nm
   > Z 2:     0.00 nm
   > Z 3:     0.00 nm
   > Z 4:   -43.34 nm
   > Z 5:   -14.84 nm
   > Z 6:     0.00 nm
   > Z 7:     1.66 nm
   > Z 8:     1.66 nm
Diagnostics : {'method': 'phase_diversity', 'cost': 0.09085813301484857, 'nfev': 11, 'status': 2, 'message': '`ftol` termination condition is satisfied.', 'psf_focus_rmse': 0.00011255992187792788, 'psf_defocus_rmse': 9.210154459556276e-05}


## Validation croisée sur les données du simulateur

On compare les coefficients de Zernike obtenus par chaque méthode à la vérité terrain (`coeffs_true_nm`, décomposition directe de la phase injectée par le simulateur). Gerchberg-Saxton étant sensible à son initialisation aléatoire, on le rejoue sur plusieurs graines pour quantifier sa variabilité plutôt que de ne montrer qu'un seul tirage.

In [7]:
def rms_error(coeffs_nm: np.ndarray) -> float:
    """RMS error [nm] against ground truth, over modes 4..zernike_order."""
    return float(np.sqrt(np.mean((coeffs_nm[3:] - coeffs_true_nm[3:]) ** 2)))


def correlation(coeffs_nm: np.ndarray) -> float:
    """Pearson correlation with ground truth, over modes 4..zernike_order."""
    return float(np.corrcoef(coeffs_true_nm[3:], coeffs_nm[3:])[0, 1])


# --- Robustesse de Gerchberg-Saxton sur plusieurs initialisations ---
gs_seeds = range(8)
gs_runs = {
    seed: psf_to_wfe(method="gerchberg_saxton", max_iter=300, seed=seed, **COMMON)[0]
    for seed in gs_seeds
}

print(f"{'seed':>6} {'RMS (nm)':>10} {'corr':>8}")
for seed, c in gs_runs.items():
    print(f"{seed:6d} {rms_error(c):10.2f} {correlation(c):8.3f}")

best_seed = min(gs_runs, key=lambda s: rms_error(gs_runs[s]))
coeffs_gs_best = gs_runs[best_seed]
print(f"\nMeilleure initialisation : seed={best_seed} "
      f"(RMS={rms_error(coeffs_gs_best):.2f} nm, corr={correlation(coeffs_gs_best):+.3f})")

2026-11-01 15:20:09.955 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:476 - Masque sur la grille de la PSF (256x256) mais ouverture détectée plus petite que le cadre (205x205 px) : recadrée/recentrée sans changement de résolution (voir REFERENCE_TECHNIQUE.md §12.3/§12.5).


2026-11-01 15:20:10.628 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:476 - Masque sur la grille de la PSF (256x256) mais ouverture détectée plus petite que le cadre (205x205 px) : recadrée/recentrée sans changement de résolution (voir REFERENCE_TECHNIQUE.md §12.3/§12.5).


2026-11-01 15:20:11.323 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:476 - Masque sur la grille de la PSF (256x256) mais ouverture détectée plus petite que le cadre (205x205 px) : recadrée/recentrée sans changement de résolution (voir REFERENCE_TECHNIQUE.md §12.3/§12.5).


2026-11-01 15:20:11.986 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:476 - Masque sur la grille de la PSF (256x256) mais ouverture détectée plus petite que le cadre (205x205 px) : recadrée/recentrée sans changement de résolution (voir REFERENCE_TECHNIQUE.md §12.3/§12.5).


2026-11-01 15:20:12.631 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:476 - Masque sur la grille de la PSF (256x256) mais ouverture détectée plus petite que le cadre (205x205 px) : recadrée/recentrée sans changement de résolution (voir REFERENCE_TECHNIQUE.md §12.3/§12.5).


2026-11-01 15:20:13.293 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:476 - Masque sur la grille de la PSF (256x256) mais ouverture détectée plus petite que le cadre (205x205 px) : recadrée/recentrée sans changement de résolution (voir REFERENCE_TECHNIQUE.md §12.3/§12.5).


2026-11-01 15:20:13.968 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:476 - Masque sur la grille de la PSF (256x256) mais ouverture détectée plus petite que le cadre (205x205 px) : recadrée/recentrée sans changement de résolution (voir REFERENCE_TECHNIQUE.md §12.3/§12.5).


2026-11-01 15:20:14.662 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:476 - Masque sur la grille de la PSF (256x256) mais ouverture détectée plus petite que le cadre (205x205 px) : recadrée/recentrée sans changement de résolution (voir REFERENCE_TECHNIQUE.md §12.3/§12.5).


  seed   RMS (nm)     corr
     0       5.48    0.873
     1      17.72   -0.186
     2       4.70    0.913
     3      22.67   -1.000
     4      20.93   -0.934
     5       5.66    0.866
     6      17.94   -0.354
     7      22.84    0.562

Meilleure initialisation : seed=2 (RMS=4.70 nm, corr=+0.913)


In [8]:
results = {
    "Gerchberg-Saxton (best of 8)": coeffs_gs_best,
    "Optimisation": coeffs_opt,
    "Diversité de phase": coeffs_pd,
}

print(f"{'Méthode':<32} {'RMS (nm)':>10} {'corr':>8}")
for name, c in results.items():
    print(f"{name:<32} {rms_error(c):10.2f} {correlation(c):8.3f}")

# --- Comparaison graphique des premiers modes ---
n_show = 8
modes = np.arange(4, n_show + 1)
mode_labels = [f"Z{m}" for m in modes]

fig = go.Figure()
fig.add_bar(x=mode_labels, y=coeffs_true_nm[3:n_show], name="Vérité terrain")
for name, c in results.items():
    fig.add_bar(x=mode_labels, y=c[3:n_show], name=name)
fig.update_layout(
    barmode="group",
    title="Comparaison des coefficients de Zernike récupérés (données simulées)",
    yaxis_title="Coefficient [nm WFE]",
    width=750, height=420,
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.add_hline(y=0, line_color="black", line_width=0.8)
fig.show()

Méthode                            RMS (nm)     corr
Gerchberg-Saxton (best of 8)           4.70    0.913
Optimisation                           4.98    0.890
Diversité de phase                    24.55   -0.989


### Interprétation

- **Gerchberg-Saxton** est le plus instable des trois : son initialisation aléatoire fait varier fortement la qualité de la reconstruction d'un tirage à l'autre (certaines graines stagnent, d'autres récupèrent la phase quasi correcte **ou son opposée**).
- **L'optimisation directe** est déterministe et récupère systématiquement la bonne *amplitude* des aberrations, mais avec une corrélation proche de **-1** avec la vérité : c'est l'ambiguïté de signe fondamentale de la récupération de phase à partir d'une seule PSF (une aberration et son opposé produisent la même image en intensité).
- **La diversité de phase** est la seule méthode qui lève cette ambiguïté grâce à la seconde PSF défocalisée : sur ces données simulées (sans bruit), elle récupère la phase **quasi exactement** (RMS ≈ 0, corrélation ≈ +1).

Ce constat confirme, a posteriori, le choix architectural : la diversité de phase n'est pas interchangeable avec les deux autres méthodes — elle répond à une limitation physique qu'elles ne peuvent pas lever, d'où sa conservation en tant que méthode à part entière (avec sa propre entrée de données) plutôt qu'une simple variante d'algorithme.

## Lever l'ambiguïté de signe : a priori sur un mode pair (Z13 / Z15)

Comme détaillé dans [`REFERENCE_TECHNIQUE.md` §3.7](REFERENCE_TECHNIQUE.md#37-lever-lambiguïté-de-signe-par-un-a-priori),
l'ambiguïté « image jumelle » d'une PSF unique n'affecte que les modes de
**fréquence azimutale paire** ($Z_4$ défocus, $Z_5$/$Z_6$ astigmatisme,
$Z_{11}$ sphérique, $Z_{12}$/$Z_{13}$ astigmatisme secondaire, …) — et ils
basculent **tous ensemble**. Les modes de fréquence **impaire** (coma
$Z_7$/$Z_8$, trèfle $Z_9$/$Z_{10}$, …) sont récupérés avec le bon signe
**dès une seule PSF**, sans aucune ambiguïté.

On le vérifie ici sur une phase synthétique multi-modes (coefficients
aléatoires sur $Z_4 \dots Z_N$, $N=13$ puis $N=15$), en utilisant le vrai
masque du simulateur et les mêmes briques que le pipeline (`optics.py` pour
générer la PSF de référence à partir de coefficients connus,
`backends.optimization` / `backends.gerchberg_saxton` + `zernike_fit` pour
la récupérer).

In [9]:
import poppy.zernike as zern
from eoqual.phaseretrieval.optics import build_zernike_basis, phase_from_coefficients, psf_from_phase
from eoqual.phaseretrieval.backends.gerchberg_saxton import RetrievalConfig, gerchberg_saxton
from eoqual.phaseretrieval.backends.optimization import OpticalConfig, fit_zernike_from_psf
from eoqual.phaseretrieval.zernike_fit import even_azimuthal_mask, resolve_sign_ambiguity

RAD_TO_NM = COMMON["wavelength"] * 1e9 / (2 * np.pi)


def synthetic_case(order: int, seed: int):
    """Build a synthetic multi-mode phase/PSF pair and its true Zernike coefficients."""
    rng = np.random.default_rng(seed)
    coeffs_nm = np.zeros(order)
    coeffs_nm[3:] = rng.uniform(-60, 60, order - 3)
    basis = build_zernike_basis(npix=256, nterms=order)
    phase_rad = phase_from_coefficients(coeffs_nm / RAD_TO_NM, basis, mask_ref)
    psf_synth = psf_from_phase(phase_rad, mask_ref)
    return coeffs_nm, psf_synth


def corr_rms(a: np.ndarray, b: np.ndarray) -> tuple:
    return float(np.corrcoef(a[3:], b[3:])[0, 1]), float(np.sqrt(np.mean((a[3:] - b[3:]) ** 2)))

In [10]:
# --- Cas de démonstration : Z4..Z15, un seul tirage ---
ORDER_DEMO = 15
true_demo_nm, psf_demo = synthetic_case(ORDER_DEMO, seed=2)

opt_cfg = OpticalConfig(wavelength=COMMON["wavelength"], zernike_order=ORDER_DEMO, verbose=False)
coeffs_raw, _ = fit_zernike_from_psf(psf_demo, mask_ref, opt_cfg)
coeffs_fixed = resolve_sign_ambiguity(coeffs_raw, reference_index=4, expected_sign=np.sign(true_demo_nm[3]))

even = even_azimuthal_mask(ORDER_DEMO)
print(f"{'j':>3} {'m parity':>9} {'true (nm)':>10} {'raw (nm)':>10} {'corrected (nm)':>15}")
for j in range(4, ORDER_DEMO + 1):
    parity = "even" if even[j-1] else "odd"
    print(f"{j:3d} {parity:>9} {true_demo_nm[j-1]:10.2f} {coeffs_raw[j-1]:10.2f} {coeffs_fixed[j-1]:15.2f}")

  j  m parity  true (nm)   raw (nm)  corrected (nm)
  4      even     -28.61     -61.82          -61.82
  5      even     -24.18      38.45           38.45
  6      even      37.71      -9.04           -9.04
  7       odd     -48.97     -30.94          -30.94
  8       odd      12.01      -2.51           -2.51
  9       odd      27.43      35.65           35.65
 10       odd     -37.45      -3.67           -3.67
 11      even     -53.38      -2.85           -2.85
 12      even     -27.00      26.41           26.41
 13      even      18.89     -10.90          -10.90
 14      even       7.47      -4.99           -4.99
 15      even     -41.99      16.00           16.00


Les modes impairs (coma, trèfle, …) ont déjà le bon signe avant même correction ; seuls les modes pairs étaient inversés, et la correction (a priori sur le seul $Z_4$) les remet tous d'aplomb simultanément.

In [11]:
# --- Robustesse statistique : 10 tirages aléatoires, ordre 13 et 15 ---
for order in (13, 15):
    print(f"\n=== zernike_order = {order} ===")
    n_ok = 0
    opt_cfg = OpticalConfig(wavelength=COMMON["wavelength"], zernike_order=order, verbose=False)
    for trial in range(10):
        true_nm, psf_synth = synthetic_case(order, seed=trial)
        coeffs_raw, diag = fit_zernike_from_psf(psf_synth, mask_ref, opt_cfg)
        coeffs_fixed = resolve_sign_ambiguity(coeffs_raw, 4, np.sign(true_nm[3]))
        corr, rms = corr_rms(coeffs_fixed, true_nm)
        ok = corr > 0.99
        n_ok += ok
        print(f"  trial={trial}  cost={diag['cost']:.1e}  corr={corr:+.3f}  RMS={rms:6.2f} nm  "
              f"{'OK' if ok else 'minimum local (nécessite un multi-start)'}")
    print(f"  -> {n_ok}/10 tirages correctement recalés après correction de signe")


=== zernike_order = 13 ===


  trial=0  cost=6.1e-02  corr=+0.285  RMS= 43.99 nm  minimum local (nécessite un multi-start)


  trial=1  cost=4.1e-02  corr=-0.466  RMS= 60.74 nm  minimum local (nécessite un multi-start)


  trial=2  cost=3.3e-02  corr=+0.204  RMS= 39.14 nm  minimum local (nécessite un multi-start)


  trial=3  cost=5.2e-02  corr=+0.420  RMS= 29.80 nm  minimum local (nécessite un multi-start)


  trial=4  cost=5.0e-02  corr=-0.083  RMS= 50.98 nm  minimum local (nécessite un multi-start)


... (11 outputs left out)

  trial=6  cost=4.6e-02  corr=+0.574  RMS= 29.65 nm  minimum local (nécessite un multi-start)


  trial=7  cost=2.8e-02  corr=+0.271  RMS= 36.34 nm  minimum local (nécessite un multi-start)


  trial=8  cost=3.5e-02  corr=-0.061  RMS= 39.49 nm  minimum local (nécessite un multi-start)


  trial=9  cost=2.1e-02  corr=-0.013  RMS= 45.29 nm  minimum local (nécessite un multi-start)
  -> 0/10 tirages correctement recalés après correction de signe


Les tirages non résolus au premier essai sont dus à une non-convergence générique du solveur (minimum local, `cost` non négligeable) — sans lien avec l'ambiguïté de signe — et se corrigent avec quelques redémarrages aléatoires de `least_squares` (voir §3.7 du document de référence).

In [12]:
# --- Même test avec Gerchberg-Saxton : la correction y est peu fiable ---
ORDER_GS = 15
true_nm, psf_synth = synthetic_case(ORDER_GS, seed=7)
zc = ZernikeConfig(order=ORDER_GS, wavelength=COMMON["wavelength"])

print(f"{'seed':>5} {'corr (brut)':>12} {'corr (corrigé)':>15}")
for seed in range(6):
    gs_cfg = RetrievalConfig(max_iter=300, seed=seed, verbose=False)
    phase_rad = gerchberg_saxton(psf_synth, mask_ref, gs_cfg)
    coeffs_raw, _, _ = fit_zernike(phase_rad, mask_ref, zc)
    coeffs_fixed = resolve_sign_ambiguity(coeffs_raw, 4, np.sign(true_nm[3]))

    corr_raw, _ = corr_rms(coeffs_raw, true_nm)
    corr_fixed, _ = corr_rms(coeffs_fixed, true_nm)
    print(f"{seed:5d} {corr_raw:+12.3f} {corr_fixed:+15.3f}")

 seed  corr (brut)  corr (corrigé)


    0       +0.671          +0.255


    1       +0.007          -0.633


    2       +0.173          +0.173


    3       -0.066          -0.596


    4       -0.455          +0.179


    5       +0.174          +0.174


Contrairement à l'optimisation directe, la correction n'améliore pas systématiquement Gerchberg-Saxton : quand celui-ci n'a pas convergé proprement vers l'une des deux branches exactes, le signe du mode de référence ($Z_4$) utilisé pour décider de la correction est lui-même peu fiable.

## Visualisation interactive (Plotly), à la demande

`psf_to_wfe(..., verbose=True)` affiche la PSF mesurée, le masque, la carte de phase (mesurée le cas échéant, et reconstruite à partir du fit Zernike) ainsi que la PSF prédite. Exemple avec la diversité de phase (la méthode la plus fidèle sur ces données).

In [13]:
_ = psf_to_wfe(
    method="phase_diversity",
    diversity_path=DATA_DIR / "psf_diversity_simu.tiff",
    defocus=275e-9,
    roi_radius=32,
    **{**COMMON, "verbose": True},
)

2026-11-01 15:21:01.589 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:476 - Masque sur la grille de la PSF (256x256) mais ouverture détectée plus petite que le cadre (205x205 px) : recadrée/recentrée sans changement de résolution (voir REFERENCE_TECHNIQUE.md §12.3/§12.5).


   Iteration     Total nfev        Cost      Cost reduction    Step norm     Optimality   
       0              1         3.3912e-01                                    7.25e-01    
       1              3         2.2695e-01      1.12e-01       1.78e-01       4.40e-01    
       2              4         9.6848e-02      1.30e-01       3.56e-01       1.64e-01    


       3              5         9.0936e-02      5.91e-03       6.17e-02       2.06e-02    


       4              6         9.0863e-02      7.33e-05       6.04e-03       5.54e-03    
       5              7         9.0859e-02      4.37e-06       1.57e-03       1.75e-03    
       6              8         9.0858e-02      4.93e-07       5.37e-04       5.54e-04    


       7              9         9.0858e-02      5.50e-08       1.83e-04       1.84e-04    


       8             10         9.0858e-02      6.33e-09       6.23e-05       6.15e-05    
       9             11         9.0858e-02      7.25e-10       2.11e-05       2.07e-05    
`ftol` termination condition is satisfied.
Function evaluations 11, initial cost 3.3912e-01, final cost 9.0858e-02, first-order optimality 2.07e-05.


## Fonctionnalités avancées

Les sections ci-dessous couvrent les évolutions du projet **non
représentées** par les données `data/*_simu.tiff` utilisées jusqu'ici
(toutes générées avec une pupille pleine non obstruée, sans WFE haut
ordre au-delà de `zernike_order`) : le module de simulation en mémoire
(`eoqual.phaseretrieval.simulation`, sans fichier à préparer), l'option
`orthogonal_basis` pour une pupille obstruée et son retour à la
nomenclature standard (`project_orthogonal_to_standard`), et
l'incorporation d'une carte de WFE haut ordre connue
(`known_ho_wfe_nm`), les a priori individuels par mode
(`zernike_priors`/`zernike_bounds`) et le suréchantillonnage en
mémoire (`aperture_px`). Voir `REFERENCE_TECHNIQUE.md` §6.7, §10,
§11.3, §6.5bis et §12.5 pour le détail théorique de chacune.
Suivent enfin deux exemples « recette » plus directs : lire une PSF
et la décomposer (pandas + plotly), et simuler/décomposer un tirage
aléatoire de Zernike avec un budget de RMS donné.

### Module de simulation : cas de test sans fichier externe

Contrairement aux sections précédentes (PSF chargées depuis
`data/*.tiff`), `eoqual.phaseretrieval.simulation` génère une PSF (ou une
paire focus/défocus, avec bruit détecteur réaliste optionnel) directement
en mémoire, à partir de coefficients de Zernike connus — pratique pour un
test rapide sans préparer de fichiers.

Point d'architecture à noter : la simulation travaille avec les fonctions
**bas niveau** de chaque backend (`fit_zernike_from_psf`,
`fit_phase_diversity`, `gerchberg_saxton` + `fit_zernike`), pas avec
`psf_to_wfe` (orienté fichiers, utilisé dans tout ce notebook jusqu'ici) :
ce sont deux points d'entrée complémentaires, pas concurrents — voir
`REFERENCE_TECHNIQUE.md` §6.2 (point d'entrée unifié) vs §6.7 (module de
simulation).

In [14]:
from eoqual.phaseretrieval.simulation import (
    simulate_psf_from_zernike,
    simulate_phase_diversity_pair,
    PhaseDiversitySimulationConfig,
    DetectorNoiseConfig,
)
from eoqual.phaseretrieval.backends.optimization import OpticalConfig, fit_zernike_from_psf
from eoqual.phaseretrieval.backends.phase_diversity import PhaseDiversityConfig, fit_phase_diversity

ADV_WAVELENGTH = 550e-9
ADV_NPIX = 128
ADV_ORDER = 12  # ordre 15 sujet à un minimum local isolé pour "optimization" à ce npix
ADV_TRUTH = {4: 55.0, 5: 27.5}  # même vérité que data/*_simu.tiff, en mémoire cette fois

# Une PSF simulée avec bruit détecteur réaliste (Poisson + lecture), méthode "optimization"
sim = simulate_psf_from_zernike(
    ADV_TRUTH, wavelength=ADV_WAVELENGTH, npix=ADV_NPIX, zernike_order=ADV_ORDER,
    noise=DetectorNoiseConfig(photon_count=1e6, read_noise_e=5.0, seed=0),
)
opt_cfg = OpticalConfig(wavelength=ADV_WAVELENGTH, zernike_order=ADV_ORDER, verbose=False)
coeffs_sim, diag_sim = fit_zernike_from_psf(sim["psf"], sim["mask"], opt_cfg)

print("Vérité simulée        : Z4=%.1f  Z5=%.1f" % (sim["coeffs_nm"][3], sim["coeffs_nm"][4]))
print("Ajustement (avec bruit): Z4=%.2f  Z5=%.2f" % (coeffs_sim[3], coeffs_sim[4]))
print("  (signe parfois inversé sur 1 PSF bruitée : ambiguïté attendue, §3.7 — comparer |corrélation|)")

# Une paire focus/défocus bruitée, méthode "phase_diversity"
pd_sim_cfg = PhaseDiversitySimulationConfig(
    wavelength=ADV_WAVELENGTH, npix=ADV_NPIX, zernike_order=ADV_ORDER, defocus=275e-9,
    noise=DetectorNoiseConfig(photon_count=1e6, read_noise_e=5.0, seed=0),
)
pair = simulate_phase_diversity_pair(ADV_TRUTH, pd_sim_cfg)
pd_cfg = PhaseDiversityConfig(
    wavelength=ADV_WAVELENGTH, zernike_order=ADV_ORDER, defocus=pair["defocus"],
    verbose=False, max_nfev=300,
)
coeffs_pair, diag_pair = fit_phase_diversity(pair["psf_focus"], pair["psf_defocus"], pair["mask"], pd_cfg)
print("Diversité de phase (avec bruit) : Z4=%.2f  Z5=%.2f" % (coeffs_pair[3], coeffs_pair[4]))

Vérité simulée        : Z4=55.0  Z5=27.5
Ajustement (avec bruit): Z4=-55.13  Z5=-28.00
  (signe parfois inversé sur 1 PSF bruitée : ambiguïté attendue, §3.7 — comparer |corrélation|)


Diversité de phase (avec bruit) : Z4=61.88  Z5=19.10


### Pupille obstruée : `orthogonal_basis` et retour à la base standard

La base de Zernike standard n'est orthogonale que sur un disque plein
(§10) : sur une pupille obstruée (obstruction centrale, bras d'araignée —
le cas courant en télédétection, §8.1), `orthogonal_basis=True` (sur
`OpticalConfig`/`PhaseDiversityConfig`, et côté simulation pour rester
cohérent — §10.1) bascule sur une base ré-orthogonalisée par Gram-Schmidt
sur le masque réel : meilleur conditionnement, mais les coefficients ne
portent plus le nom usuel (« défocus », « astigmatisme »…) au-delà des
tout premiers ordres. `zernike_fit.project_orthogonal_to_standard`
reconvertit après coup vers la nomenclature standard pour
l'interprétation (§10.2) — au prix d'un léger écart quand la vérité
d'origine n'est pas exactement définie dans cette même base standard
(mélange de piston, voir §10.2).

In [15]:
from eoqual.phaseretrieval.pupil import PupilGenerator
from eoqual.phaseretrieval.zernike_basis_orthogonal import resolve_zernike_basis
from eoqual.phaseretrieval.zernike_fit import project_orthogonal_to_standard

obstructed_mask = PupilGenerator(ADV_NPIX).annular(obscuration=0.3)  # 30 % d'obstruction centrale

# Vérité pensée en base standard (aberration physique connue), ajustée en base Gram-Schmidt
sim_obstr = simulate_psf_from_zernike(
    ADV_TRUTH, wavelength=ADV_WAVELENGTH, npix=ADV_NPIX, zernike_order=12, mask=obstructed_mask,
)
opt_cfg_gs = OpticalConfig(wavelength=ADV_WAVELENGTH, zernike_order=12, verbose=False, orthogonal_basis=True)
coeffs_gs_basis, diag_gs_basis = fit_zernike_from_psf(sim_obstr["psf"], obstructed_mask, opt_cfg_gs)

# Retour à la base standard pour l'interprétation
basis_gs = resolve_zernike_basis(npix=ADV_NPIX, nterms=12, mask=obstructed_mask, orthogonalize=True)
coeffs_std, phase_std_nm, diag_reproj = project_orthogonal_to_standard(
    coeffs_gs_basis, basis_gs, obstructed_mask, wavelength=ADV_WAVELENGTH,
)

print("Vérité (base standard)              : Z4=55.0  Z5=27.5")
print("Ajustement Gram-Schmidt, reprojeté  : Z4=%.2f  Z5=%.2f" % (coeffs_std[3], coeffs_std[4]))
print("condition_number (base standard, pupille obstruée) : %.2f" % diag_reproj["condition_number"])

Vérité (base standard)              : Z4=55.0  Z5=27.5
Ajustement Gram-Schmidt, reprojeté  : Z4=55.03  Z5=27.46
condition_number (base standard, pupille obstruée) : 1.54


### WFE haut ordre connue (`known_ho_wfe_nm`)

Une carte de WFE mesurée indépendamment (interférogramme, as-built,
calibration sol), au-delà de `zernike_order`, biaise les coefficients bas
ordre si elle n'est pas prise en compte dans le modèle (§11.2 :
identifiabilité). `known_ho_wfe_nm` (`OpticalConfig`/`PhaseDiversityConfig`,
ou `psf_to_wfe(..., known_ho_wfe_path=...)`/`--known_ho_wfe` en CLI pour
une carte sur fichier) l'ajoute au modèle direct sans jamais l'ajuster —
§11.3. Démonstration avec un unique mode $Z_{21}$ connu de 40 nm, hors de
portée de `zernike_order=12` ci-dessous.

In [16]:
from eoqual.phaseretrieval.optics import build_zernike_basis, phase_from_coefficients

full_disk_mask = PupilGenerator(ADV_NPIX).simple_disk()
rad_to_nm_adv = ADV_WAVELENGTH * 1e9 / (2 * np.pi)
basis25 = build_zernike_basis(npix=ADV_NPIX, nterms=25)
ho_coeffs = np.zeros(25)
ho_coeffs[20] = 40.0  # Z21, nm — hors de portée de zernike_order=12 ci-dessous
ho_wfe_nm = phase_from_coefficients(ho_coeffs / rad_to_nm_adv, basis25, full_disk_mask) * rad_to_nm_adv

sim_ho = simulate_psf_from_zernike(
    ADV_TRUTH, wavelength=ADV_WAVELENGTH, npix=ADV_NPIX, zernike_order=12,
    mask=full_disk_mask, known_ho_wfe_nm=ho_wfe_nm,
)

coeffs_biased, diag_biased = fit_zernike_from_psf(
    sim_ho["psf"], full_disk_mask,
    OpticalConfig(wavelength=ADV_WAVELENGTH, zernike_order=12, verbose=False),
)
coeffs_corrected, diag_corrected = fit_zernike_from_psf(
    sim_ho["psf"], full_disk_mask,
    OpticalConfig(wavelength=ADV_WAVELENGTH, zernike_order=12, verbose=False, known_ho_wfe_nm=ho_wfe_nm),
)

print("Vérité                    : Z4=55.0  Z5=27.5")
print("Sans la carte HO connue   : Z4=%.2f  Z5=%.2f  (biaisé)" % (coeffs_biased[3], coeffs_biased[4]))
print("Avec la carte HO connue   : Z4=%.2f  Z5=%.2f" % (coeffs_corrected[3], coeffs_corrected[4]))

Vérité                    : Z4=55.0  Z5=27.5
Sans la carte HO connue   : Z4=44.73  Z5=41.51  (biaisé)
Avec la carte HO connue   : Z4=55.00  Z5=27.50


### A priori individuels par mode : `zernike_priors` / `zernike_bounds`

`OpticalConfig.l2_reg` (déjà vu plus haut) est un cas particulier d'un
mécanisme plus général : un a priori bayésien **individuel** par mode
(moyenne et poids propres à chaque coefficient) plutôt qu'un unique poids
vers zéro pour tous les modes — voir `REFERENCE_TECHNIQUE.md` §6.5bis
pour la dérivation complète (MAP gaussien = pénalité L2, MAP uniforme =
borne dure ; `zernike_priors` en est la forme souple, `zernike_bounds` la
forme dure). Démonstration ci-dessous : à l'ordre 12, sans a priori, la
reconstruction mono-PSF souffre de l'ambiguïté de signe des modes pairs
(§3.7 — $Z_4$, $Z_6$ inversés ; $Z_8$, impair, jamais ambigu) ; un a
priori individuel fort et correct sur les modes concernés la résout tout
en la débiaisant, **sans** passer par une seconde image de diversité.

**Attention à l'origine d'un a priori dans ce contexte (télédétection,
§8)** : une décroissance systématique de l'amplitude avec l'ordre radial
(courante en optique adaptative sol, issue de la statistique de turbulence
de Kolmogorov) n'est *pas*, en général, justifiée pour la WFE propre d'un
instrument spatial — voir la mise en garde de §6.5bis. Les valeurs
ci-dessous sont des a priori "connus" arbitraires à des fins de
démonstration, pas une telle hypothèse générique.

In [17]:
PRIOR_TRUTH = {4: 60.0, 6: 35.0, 8: -25.0}  # Z4, Z6 pairs (ambigus sur 1 PSF) ; Z8 impair (jamais ambigu)
sim_prior = simulate_psf_from_zernike(PRIOR_TRUTH, wavelength=ADV_WAVELENGTH, npix=ADV_NPIX, zernike_order=ADV_ORDER)

coeffs_noprior, _ = fit_zernike_from_psf(
    sim_prior["psf"], sim_prior["mask"],
    OpticalConfig(wavelength=ADV_WAVELENGTH, zernike_order=ADV_ORDER, verbose=False),
)

# A priori individuel fort (petit écart-type) et correct sur les modes pairs concernés
priors = {4: (60.0, 2.0), 6: (35.0, 2.0)}
coeffs_prior, _ = fit_zernike_from_psf(
    sim_prior["psf"], sim_prior["mask"],
    OpticalConfig(wavelength=ADV_WAVELENGTH, zernike_order=ADV_ORDER, verbose=False, zernike_priors=priors),
)

print("V\u00e9rit\u00e9                         : Z4=%.1f  Z6=%.1f  Z8=%.1f" % (PRIOR_TRUTH[4], PRIOR_TRUTH[6], PRIOR_TRUTH[8]))
print("Sans a priori (1 PSF)          : Z4=%.2f  Z6=%.2f  Z8=%.2f" % (coeffs_noprior[3], coeffs_noprior[5], coeffs_noprior[7]))
print("  (signe de Z4/Z6 parfois invers\u00e9 : ambig\u00fcit\u00e9 des modes pairs, \u00a73.7)")
print("Avec a priori individuel Z4,Z6 : Z4=%.2f  Z6=%.2f  Z8=%.2f" % (coeffs_prior[3], coeffs_prior[5], coeffs_prior[7]))

# Contrainte physique (borne dure) : Z4 jamais négatif dans ce montage
coeffs_bounded, _ = fit_zernike_from_psf(
    sim_prior["psf"], sim_prior["mask"],
    OpticalConfig(wavelength=ADV_WAVELENGTH, zernike_order=ADV_ORDER, verbose=False, zernike_bounds={4: (0.0, 500.0)}),
)
print("Avec borne Z4 in [0, 500] nm   : Z4=%.2f" % coeffs_bounded[3])

Vérité                         : Z4=60.0  Z6=35.0  Z8=-25.0
Sans a priori (1 PSF)          : Z4=-59.65  Z6=-35.92  Z8=-26.86
  (signe de Z4/Z6 parfois inversé : ambigüité des modes pairs, §3.7)
Avec a priori individuel Z4,Z6 : Z4=60.00  Z6=35.00  Z8=-25.00


Avec borne Z4 in [0, 500] nm   : Z4=60.00


### Suréchantillonnage en mémoire : `aperture_px`

`PupilGenerator(npix, aperture_px=...)` et
`simulate_psf_from_zernike(..., aperture_px=...)` placent l'ouverture sur
seulement `aperture_px` pixels à l'intérieur d'un cadre `npix x npix`
complété de zéros — le même suréchantillonnage que `pixel_pitch`/
`oversampling` (§12.1) apportent au pipeline fichier, mais ici directement
en mémoire. **Important** : la base de Zernike doit recevoir le même
`aperture_px`, sinon elle reste évaluée comme si l'ouverture remplissait
tout le cadre — un décalage d'échelle silencieux corrigé en §12.5 de
`REFERENCE_TECHNIQUE.md` (`simulate_psf_from_zernike`/`OpticalConfig` s'en
chargent automatiquement dès que le même `aperture_px` est passé aux
deux). Démontré ci-dessous avec `aperture_px = npix/2`, l'échantillonnage
de Nyquist exact (`fc = 1.0`, §9/§12.5) — un bon cas pour visualiser la
FTM et son repère `fc`.

In [18]:
APERTURE_NPIX, APERTURE_PX = 128, 64  # ouverture = moitié du cadre (facteur 2)

sim_oversampled = simulate_psf_from_zernike(
    {4: 55.0, 5: 27.5}, wavelength=COMMON["wavelength"], npix=APERTURE_NPIX,
    zernike_order=ADV_ORDER, aperture_px=APERTURE_PX,
)
print(f"Pixels illuminés : {int(sim_oversampled['mask'].sum())} / {APERTURE_NPIX**2} "
      f"(disque de diamètre {APERTURE_PX} dans un cadre {APERTURE_NPIX})")

opt_cfg_over = OpticalConfig(
    wavelength=COMMON["wavelength"], zernike_order=ADV_ORDER, verbose=False, aperture_px=APERTURE_PX,
)
coeffs_over, diag_over = fit_zernike_from_psf(sim_oversampled["psf"], sim_oversampled["mask"], opt_cfg_over)
print(f"Vérité       : Z4={sim_oversampled['coeffs_nm'][3]:.2f}  Z5={sim_oversampled['coeffs_nm'][4]:.2f}")
print(f"Récupéré     : Z4={coeffs_over[3]:.2f}  Z5={coeffs_over[4]:.2f}  (coût={diag_over['cost']:.2e})")

import pandas as pd

from eoqual.phaseretrieval.diagnostics import (
    compute_mtf, encircled_energy, fwhm, mtf_cutoff_frequency, mtf_frequency_axis,
    mtf_cuts, mtf_integral, mtf_value_at, strehl_ratio,
)
from eoqual.phaseretrieval.io_utils import plot_mtf_cuts

# FTM, coupes H/V, référence vs estimée -- avec deux repères : fc (rouge)
# et fe/2 (bleu, Nyquist). APERTURE_PX = npix/2 correspond exactement à
# l'échantillonnage de Nyquist (fc = 0,5, pile sur fe/2) : le cas le mieux
# choisi pour illustrer ces deux repères confondus.
basis_over = build_zernike_basis(npix=APERTURE_NPIX, nterms=ADV_ORDER, aperture_px=APERTURE_PX)
rad_to_nm_over = COMMON["wavelength"] * 1e9 / (2 * np.pi)
phase_over_rad = phase_from_coefficients(coeffs_over / rad_to_nm_over, basis_over, sim_oversampled["mask"])
psf_over_estimee = psf_from_phase(phase_over_rad, sim_oversampled["mask"])

mtf_truth = compute_mtf(sim_oversampled["psf_truth"])
mtf_estimee = compute_mtf(psf_over_estimee)
fc_over = mtf_cutoff_frequency(APERTURE_NPIX, APERTURE_PX)
print(f"fc (fraction de fe) : {fc_over:.2f}")

plot_mtf_cuts(mtf_estimee, mtf_reference=mtf_truth, fc=fc_over, title="FTM — vérité vs récupérée").show()

# Tableau pandas : valeurs de FTM à fe/4, fe/2, et intégrale entre 0 et fe/2.
freq_over = mtf_frequency_axis(APERTURE_NPIX)
cuts_over = {
    "Vérité — coupe H": mtf_cuts(mtf_truth)[0],
    "Vérité — coupe V": mtf_cuts(mtf_truth)[1],
    "Récupérée — coupe H": mtf_cuts(mtf_estimee)[0],
    "Récupérée — coupe V": mtf_cuts(mtf_estimee)[1],
}
df_mtf_over = pd.DataFrame({
    "Coupe": list(cuts_over.keys()),
    "FTM(fe/4)": [mtf_value_at(freq_over, c, 0.25) for c in cuts_over.values()],
    "FTM(fe/2)": [mtf_value_at(freq_over, c, 0.5) for c in cuts_over.values()],
    "Intégrale [0, fe/2]": [mtf_integral(freq_over, c, f_max=0.5) for c in cuts_over.values()],
})
print(df_mtf_over.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

# Métriques PSF : Strehl, FWHM (coupes H/V), énergie encerclée -- simulée
# (vérité) vs restituée (récupérée) -- délèguent à eoqual-core
# (eoqual.core.psf/edge) via diagnostics.py, voir REFERENCE_TECHNIQUE.md §9/§9.5.
cy_over, cx_over = APERTURE_NPIX // 2, APERTURE_NPIX // 2
psf_ideal_over = psf_from_phase(np.zeros((APERTURE_NPIX, APERTURE_NPIX)), sim_oversampled["mask"])
x_px_over = np.arange(APERTURE_NPIX) - cy_over

_, ee_truth_over = encircled_energy(sim_oversampled["psf_truth"], radii=[2.0, 5.0])
_, ee_est_over = encircled_energy(psf_over_estimee, radii=[2.0, 5.0])

df_psf_metrics_over = pd.DataFrame({
    "Métrique": ["Strehl", "FWHM H [px]", "FWHM V [px]", "EE(r=2px)", "EE(r=5px)"],
    "Simulée (vérité)": [
        strehl_ratio(sim_oversampled["psf_truth"], psf_ideal_over),
        fwhm(x_px_over, sim_oversampled["psf_truth"][cy_over, :]),
        fwhm(x_px_over, sim_oversampled["psf_truth"][:, cx_over]),
        ee_truth_over[0], ee_truth_over[1],
    ],
    "Restituée (récupérée)": [
        strehl_ratio(psf_over_estimee, psf_ideal_over),
        fwhm(x_px_over, psf_over_estimee[cy_over, :]),
        fwhm(x_px_over, psf_over_estimee[:, cx_over]),
        ee_est_over[0], ee_est_over[1],
    ],
})
print(df_psf_metrics_over.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

Pixels illuminés : 3209 / 16384 (disque de diamètre 64 dans un cadre 128)
Vérité       : Z4=55.00  Z5=27.50
Récupéré     : Z4=55.00  Z5=27.50  (coût=3.01e-25)


fc (fraction de fe) : 0.50


              Coupe  FTM(fe/4)  FTM(fe/2)  Intégrale [0, fe/2]
   Vérité — coupe H      0.162      0.001                0.146
   Vérité — coupe V      0.162      0.001                0.146
Récupérée — coupe H      0.162      0.001                0.146
Récupérée — coupe V      0.162      0.001                0.146
   Métrique  Simulée (vérité)  Restituée (récupérée)
     Strehl             0.604                  0.604
FWHM H [px]             2.194                  2.194
FWHM V [px]             2.194                  2.194
  EE(r=2px)             0.536                  0.536
  EE(r=5px)             0.881                  0.881


### Exemple : lire une PSF (imageio) et la décomposer en Zernike (pandas + plotly)

Recette minimale, de bout en bout : lire un fichier PSF avec `imageio`,
l'ajuster (`fit_zernike_from_psf`, méthode `"optimization"`, 1 PSF),
afficher les coefficients non nuls avec `pandas`, puis comparer — en 2D et
par coupes horizontale/verticale — la PSF mesurée à sa reconstruction à
partir des seuls coefficients de Zernike récupérés, ainsi que leurs FTM
(coupes H/V, axe en fraction de `fe` — `fe/2` en bleu, `fc` en rouge) et
des tableaux `pandas` récapitulatifs : valeurs de FTM à `fe/4`/`fe/2` et
intégrale `[0, fe/2]`, puis métriques PSF (Strehl, FWHM, énergie
encerclée — `eoqual-core`, §9/§9.5).

In [19]:
import imageio.v3 as iio
import pandas as pd
from plotly.subplots import make_subplots

from eoqual.phaseretrieval.backends.optimization import OpticalConfig, fit_zernike_from_psf
from eoqual.phaseretrieval.optics import build_zernike_basis, phase_from_coefficients, psf_from_phase
from eoqual.phaseretrieval.zernike_fit import resolve_sign_ambiguity
from eoqual.phaseretrieval.diagnostics import (
    compute_mtf, encircled_energy, fwhm, mtf_cutoff_frequency, mtf_frequency_axis,
    mtf_cuts, mtf_integral, mtf_value_at, strehl_ratio,
)
from eoqual.phaseretrieval.io_utils import plot_mtf_cuts

# 1) Lecture directe (imageio) + normalisation énergie (somme = 1)
psf_raw = iio.imread(DATA_DIR / "psf_simu.tiff").astype(float)
psf_measured = psf_raw / psf_raw.sum()

# 2) Décomposition en Zernike (masque déjà préparé plus haut : mask_ref)
opt_cfg_demo1 = OpticalConfig(
    wavelength=COMMON["wavelength"], zernike_order=COMMON["zernike_order"], verbose=False,
)
coeffs_demo1_nm, diag_demo1 = fit_zernike_from_psf(psf_measured, mask_ref, opt_cfg_demo1)

# 2bis) Ambiguïté de signe (§3.7) : une seule PSF ne fixe jamais le signe
# des modes pairs (défocus, astigmatisme, sphérique...) de façon fiable.
# Si le signe d'UN SEUL mode pair est connu a priori -- ici Z4, par le sens
# d'un défocus volontairement introduit -- resolve_sign_ambiguity lève
# l'ambiguïté pour TOUS les modes pairs à la fois (démonstration détaillée
# dans la section "Lever l'ambiguïté de signe" plus haut). Pour le
# pipeline fichier (psf_to_wfe), le même réglage existe directement via
# sign_prior=(4, +1.0).
coeffs_demo1_nm = resolve_sign_ambiguity(coeffs_demo1_nm, reference_index=4, expected_sign=+1.0)

# 3) Tableau des coefficients non nuls (pandas)
df_coeffs = pd.DataFrame({
    "Noll": np.arange(1, len(coeffs_demo1_nm) + 1),
    "Coefficient [nm WFE]": coeffs_demo1_nm,
})
df_coeffs = df_coeffs[df_coeffs["Coefficient [nm WFE]"].abs() > 1e-6].reset_index(drop=True)
print(df_coeffs.to_string(index=False, float_format=lambda v: f"{v:+.2f}"))
print("(signe de Z4, et donc des autres modes pairs, fixé a priori via resolve_sign_ambiguity, §3.7)")

# 4) Reconstruction de la PSF à partir des coefficients récupérés
basis_demo1 = build_zernike_basis(npix=psf_measured.shape[0], nterms=COMMON["zernike_order"])
rad_to_nm_demo1 = COMMON["wavelength"] * 1e9 / (2 * np.pi)
phase_fit_rad_demo1 = phase_from_coefficients(coeffs_demo1_nm / rad_to_nm_demo1, basis_demo1, mask_ref)
psf_reconstructed = psf_from_phase(phase_fit_rad_demo1, mask_ref)

# 5) Comparaison 2D + coupes H/V (Plotly)
n_demo1 = psf_measured.shape[0]
cy, cx = n_demo1 // 2, n_demo1 // 2
half = 24  # fenêtre de visualisation autour du pic


def _crop(img):
    return img[cy - half:cy + half, cx - half:cx + half]


x_axis = np.arange(-half, half)

fig_demo1 = make_subplots(
    rows=2, cols=2,
    subplot_titles=("PSF mesurée", "PSF reconstruite (Zernike)", "Coupe horizontale", "Coupe verticale"),
    row_heights=[0.6, 0.4],
)
fig_demo1.add_heatmap(z=_crop(psf_measured), colorscale="Inferno", showscale=False, row=1, col=1)
fig_demo1.add_heatmap(z=_crop(psf_reconstructed), colorscale="Inferno", showscale=False, row=1, col=2)

fig_demo1.add_scatter(
    x=x_axis, y=psf_measured[cy, cx - half:cx + half], name="mesurée",
    line=dict(color="black"), row=2, col=1,
)
fig_demo1.add_scatter(
    x=x_axis, y=psf_reconstructed[cy, cx - half:cx + half], name="reconstruite",
    line=dict(color="crimson", dash="dash"), row=2, col=1,
)
fig_demo1.add_scatter(
    x=x_axis, y=psf_measured[cy - half:cy + half, cx], name="mesurée",
    line=dict(color="black"), showlegend=False, row=2, col=2,
)
fig_demo1.add_scatter(
    x=x_axis, y=psf_reconstructed[cy - half:cy + half, cx], name="reconstruite",
    line=dict(color="crimson", dash="dash"), showlegend=False, row=2, col=2,
)
fig_demo1.update_layout(
    height=650, width=780,
    title="PSF mesurée vs reconstruite à partir des coefficients de Zernike",
    margin=dict(l=0, r=0, t=60, b=0),
)
fig_demo1.show()

# 6) FTM (coupes H/V), mesurée vs reconstruite -- avec fc si les
# paramètres physiques sont disponibles. Ici mask_ref remplit tout le
# cadre (n_pup_target_ref vaut None, cf. la cellule "Données du
# simulateur") : fc en ressort à 1.0, hors de l'intervalle [0, 0,5] de
# Nyquist -- ce modèle PSF brut, sans marge de suréchantillonnage,
# échantillonne en-deçà du critère de Nyquist pour la diffraction (voir
# la démonstration à fc=0,5 dans "Suréchantillonnage en mémoire" plus
# haut pour le cas correctement échantillonné).
mtf_measured = compute_mtf(psf_measured)
mtf_reconstructed = compute_mtf(psf_reconstructed)
fc_demo1 = mtf_cutoff_frequency(n_demo1, n_pup_target_ref)
print(f"fc (fraction de fe) : {fc_demo1:.2f}")

plot_mtf_cuts(
    mtf_reconstructed, mtf_reference=mtf_measured, fc=fc_demo1,
    title="FTM — mesurée vs reconstruite",
).show()

# Tableau pandas : valeurs de FTM à fe/4, fe/2, et intégrale entre 0 et fe/2.
freq_demo1 = mtf_frequency_axis(n_demo1)
cuts_demo1 = {
    "Mesurée — coupe H": mtf_cuts(mtf_measured)[0],
    "Mesurée — coupe V": mtf_cuts(mtf_measured)[1],
    "Reconstruite — coupe H": mtf_cuts(mtf_reconstructed)[0],
    "Reconstruite — coupe V": mtf_cuts(mtf_reconstructed)[1],
}
df_mtf_demo1 = pd.DataFrame({
    "Coupe": list(cuts_demo1.keys()),
    "FTM(fe/4)": [mtf_value_at(freq_demo1, c, 0.25) for c in cuts_demo1.values()],
    "FTM(fe/2)": [mtf_value_at(freq_demo1, c, 0.5) for c in cuts_demo1.values()],
    "Intégrale [0, fe/2]": [mtf_integral(freq_demo1, c, f_max=0.5) for c in cuts_demo1.values()],
})
print(df_mtf_demo1.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

# Métriques PSF : Strehl, FWHM (coupes H/V), énergie encerclée -- mesurée
# vs reconstruite (voir REFERENCE_TECHNIQUE.md §9/§9.5).
psf_ideal_demo1 = psf_from_phase(np.zeros((n_demo1, n_demo1)), mask_ref)
_, ee_measured_demo1 = encircled_energy(psf_measured, radii=[2.0, 5.0])
_, ee_reconstructed_demo1 = encircled_energy(psf_reconstructed, radii=[2.0, 5.0])

df_psf_metrics_demo1 = pd.DataFrame({
    "Métrique": ["Strehl", "FWHM H [px]", "FWHM V [px]", "EE(r=2px)", "EE(r=5px)"],
    "Mesurée": [
        strehl_ratio(psf_measured, psf_ideal_demo1),
        fwhm(x_axis, psf_measured[cy, cx - half:cx + half]),
        fwhm(x_axis, psf_measured[cy - half:cy + half, cx]),
        ee_measured_demo1[0], ee_measured_demo1[1],
    ],
    "Reconstruite": [
        strehl_ratio(psf_reconstructed, psf_ideal_demo1),
        fwhm(x_axis, psf_reconstructed[cy, cx - half:cx + half]),
        fwhm(x_axis, psf_reconstructed[cy - half:cy + half, cx]),
        ee_reconstructed_demo1[0], ee_reconstructed_demo1[1],
    ],
})
print(df_psf_metrics_demo1.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

 Noll  Coefficient [nm WFE]
    4                +26.65
    5                +15.92
    7                 -0.88
    8                 -0.88
    9                 -0.47
   10                 +0.47
   11                 -6.87
   13                 -8.27
   14                +10.19
(signe de Z4, et donc des autres modes pairs, fixé a priori via resolve_sign_ambiguity, §3.7)


fc (fraction de fe) : 0.80


                 Coupe  FTM(fe/4)  FTM(fe/2)  Intégrale [0, fe/2]
     Mesurée — coupe H      0.503      0.376                0.288
     Mesurée — coupe V      0.503      0.376                0.288
Reconstruite — coupe H      0.549      0.417                0.305
Reconstruite — coupe V      0.549      0.417                0.305
   Métrique  Mesurée  Reconstruite
     Strehl    0.815         0.890
FWHM H [px]    1.213         1.197
FWHM V [px]    1.213         1.197
  EE(r=2px)    0.776         0.809
  EE(r=5px)    0.945         0.947


### Exemple : PSF simulée à partir d'un tirage aléatoire de Zernike (budget RMS, décroissance radiale réglable)

Génère un jeu de coefficients de Zernike aléatoires — nombre de modes,
budget de RMS WFE total et décroissance d'amplitude avec l'ordre radial
tous réglables — simule la PSF correspondante, la redécompose, puis
compare la carte de WFE récupérée à la vérité (2D + RMSE), ainsi que les
FTM (coupes H/V, `fe/2` en bleu, `fc` en rouge) et les métriques PSF
(Strehl, FWHM, énergie encerclée — `eoqual-core`, §9/§9.5) de la PSF
simulée et de la PSF récupérée.

**Attention** sur la décroissance radiale : c'est un réglage pratique
pour varier les cas de test, pas une hypothèse physique sur la WFE propre
d'un instrument — une décroissance systématique avec l'ordre radial est
une propriété de la turbulence atmosphérique (Kolmogorov), pas en général
de l'aberration statique d'un instrument spatial (voir §6.5bis de
`REFERENCE_TECHNIQUE.md`). `decay=0` donne un spectre plat, sans parti
pris.

In [20]:
import poppy.zernike as zern

from eoqual.phaseretrieval.zernike_fit import resolve_sign_ambiguity
from eoqual.phaseretrieval.diagnostics import (
    compute_mtf, encircled_energy, fwhm, mtf_cutoff_frequency, mtf_frequency_axis,
    mtf_cuts, mtf_integral, mtf_value_at, strehl_ratio,
)
from eoqual.phaseretrieval.io_utils import plot_mtf_cuts


def random_decreasing_zernike(n_modes: int, rms_nm: float, decay: float = 1.0, seed: int = None) -> dict:
    """
    Tirage aléatoire de ``n_modes`` coefficients de Zernike (Noll 4..3+n_modes,
    piston/tip/tilt exclus), d'amplitude pondérée par ``(ordre radial + 1)**-decay``
    (``decay=0`` : spectre plat ; ``decay`` croissant : décroissance de plus
    en plus marquée avec l'ordre), puis renormalisé pour que la RMS WFE
    totale (sur une pupille pleine, modes de Noll orthonormés) vaille
    ``rms_nm``.
    """
    rng = np.random.default_rng(seed)
    noll_indices = np.arange(4, 4 + n_modes)
    radial_order = np.array([zern.noll_indices(int(j))[0] for j in noll_indices])
    weights = (radial_order + 1.0) ** (-decay)
    raw = rng.normal(size=n_modes) * weights
    raw *= rms_nm / np.sqrt(np.sum(raw ** 2))  # RMS totale = somme quadratique (modes orthonormés)
    return dict(zip(noll_indices.tolist(), raw.tolist()))


# --- Paramètres réglables ---
RANDOM_N_MODES = 9      # nombre de modes libres (Z4..Z(3+N_MODES))
RANDOM_RMS_NM = 80.0    # budget de RMS WFE total [nm]
RANDOM_DECAY = 1.2      # 0 = plat, >0 = décroissant avec l'ordre radial
RANDOM_ORDER = 3 + RANDOM_N_MODES

truth_coeffs_random = random_decreasing_zernike(
    RANDOM_N_MODES, RANDOM_RMS_NM, decay=RANDOM_DECAY, seed=0,
)
sim_random = simulate_psf_from_zernike(
    truth_coeffs_random, wavelength=COMMON["wavelength"], npix=ADV_NPIX, zernike_order=RANDOM_ORDER,
)

# Décomposition + levée de l'ambiguïté de signe (a priori sur Z4, comme un
# défocus dont le sens est connu — §3.7)
opt_cfg_random = OpticalConfig(wavelength=COMMON["wavelength"], zernike_order=RANDOM_ORDER, verbose=False)
coeffs_random_nm, diag_random = fit_zernike_from_psf(sim_random["psf"], sim_random["mask"], opt_cfg_random)
coeffs_random_nm = resolve_sign_ambiguity(
    coeffs_random_nm, reference_index=4,
    expected_sign=np.sign(sim_random["coeffs_nm"][3]) or 1.0,
)

# Cartes de WFE [nm] pour la comparaison 2D
basis_random = build_zernike_basis(npix=ADV_NPIX, nterms=RANDOM_ORDER)
rad_to_nm_random = COMMON["wavelength"] * 1e9 / (2 * np.pi)
wfe_truth_nm = phase_from_coefficients(
    sim_random["coeffs_nm"] / rad_to_nm_random, basis_random, sim_random["mask"],
) * rad_to_nm_random
wfe_recovered_nm = phase_from_coefficients(
    coeffs_random_nm / rad_to_nm_random, basis_random, sim_random["mask"],
) * rad_to_nm_random

inside_random = sim_random["mask"] > 0
wfe_rmse_nm = float(np.sqrt(np.mean((wfe_recovered_nm[inside_random] - wfe_truth_nm[inside_random]) ** 2)))
rms_truth_check = float(np.sqrt(np.mean(wfe_truth_nm[inside_random] ** 2)))
print(f"RMS WFE simulée (vérification) : {rms_truth_check:.2f} nm (budget demandé : {RANDOM_RMS_NM:.1f} nm)")
print(f"RMSE carte de WFE récupérée vs vérité : {wfe_rmse_nm:.2f} nm")

fig_demo2 = make_subplots(
    rows=1, cols=3,
    subplot_titles=("WFE vérité [nm]", "WFE récupérée [nm]", "Résidu [nm]"),
    horizontal_spacing=0.12,
)
_wfe_nan = lambda a: np.where(inside_random, a, np.nan)  # noqa: E731 - lisibilité locale
wfe_residual_nm = wfe_recovered_nm - wfe_truth_nm

# Deux échelles de couleur indépendantes : vérité/récupérée partagent la
# même (comparables entre elles), le résidu a la sienne -- sa dynamique
# est bien plus petite, un résidu sur l'échelle de la WFE serait écrasé
# et illisible (quasi uniforme).
wfe_absmax = float(np.nanmax(np.abs(_wfe_nan(wfe_truth_nm))))
res_absmax = float(np.nanmax(np.abs(_wfe_nan(wfe_residual_nm)))) or 1e-9

fig_demo2.add_heatmap(
    z=_wfe_nan(wfe_truth_nm), colorscale="RdBu", zmid=0, zmin=-wfe_absmax, zmax=wfe_absmax,
    colorbar=dict(title="WFE [nm]", x=0.635, len=0.9, thickness=14),
    row=1, col=1,
)
fig_demo2.add_heatmap(
    z=_wfe_nan(wfe_recovered_nm), colorscale="RdBu", zmid=0, zmin=-wfe_absmax, zmax=wfe_absmax,
    showscale=False,  # même échelle que "WFE vérité" -> une seule barre suffit
    row=1, col=2,
)
fig_demo2.add_heatmap(
    z=_wfe_nan(wfe_residual_nm), colorscale="RdBu", zmid=0, zmin=-res_absmax, zmax=res_absmax,
    colorbar=dict(title="Résidu [nm]", x=1.02, len=0.9, thickness=14),
    row=1, col=3,
)

# Ratio d'aspect 1:1 sur chaque sous-graphe (sinon les cartes, carrées, sont
# étirées pour remplir chaque case).
for col in (1, 2, 3):
    fig_demo2.update_yaxes(scaleanchor=f"x{col if col > 1 else ''}", scaleratio=1, row=1, col=col)

fig_demo2.update_layout(
    height=380, width=950,
    title=f"WFE simulée vs récupérée (ordre {RANDOM_ORDER}, RMSE={wfe_rmse_nm:.2f} nm)",
    margin=dict(l=0, r=0, t=60, b=0),
)
fig_demo2.show()

# FTM (coupes H/V), simulée vs récupérée -- avec fc (rouge) et fe/2
# (bleu). Ici le masque remplit tout le cadre (aperture_px par défaut) :
# fc = 1.0, hors de [0, 0,5] -- même régime sous-échantillonné que
# l'Exemple 1 (voir "Suréchantillonnage en mémoire" plus haut pour le cas
# correctement échantillonné, fc = 0,5).
psf_recovered_random = psf_from_phase(wfe_recovered_nm / rad_to_nm_random, sim_random["mask"])

mtf_truth_random = compute_mtf(sim_random["psf_truth"])
mtf_recovered_random = compute_mtf(psf_recovered_random)
fc_random = mtf_cutoff_frequency(ADV_NPIX, None)
print(f"fc (fraction de fe) : {fc_random:.2f}")

plot_mtf_cuts(
    mtf_recovered_random, mtf_reference=mtf_truth_random, fc=fc_random,
    title="FTM — simulée vs récupérée",
).show()

# Tableau pandas : valeurs de FTM à fe/4, fe/2, et intégrale entre 0 et fe/2.
freq_random = mtf_frequency_axis(ADV_NPIX)
cuts_random = {
    "Simulée — coupe H": mtf_cuts(mtf_truth_random)[0],
    "Simulée — coupe V": mtf_cuts(mtf_truth_random)[1],
    "Récupérée — coupe H": mtf_cuts(mtf_recovered_random)[0],
    "Récupérée — coupe V": mtf_cuts(mtf_recovered_random)[1],
}
df_mtf_random = pd.DataFrame({
    "Coupe": list(cuts_random.keys()),
    "FTM(fe/4)": [mtf_value_at(freq_random, c, 0.25) for c in cuts_random.values()],
    "FTM(fe/2)": [mtf_value_at(freq_random, c, 0.5) for c in cuts_random.values()],
    "Intégrale [0, fe/2]": [mtf_integral(freq_random, c, f_max=0.5) for c in cuts_random.values()],
})
print(df_mtf_random.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

# Métriques PSF : Strehl, FWHM (coupes H/V), énergie encerclée -- simulée
# (vérité) vs récupérée (voir REFERENCE_TECHNIQUE.md §9/§9.5).
cy_random, cx_random = ADV_NPIX // 2, ADV_NPIX // 2
x_px_random = np.arange(ADV_NPIX) - cy_random
psf_ideal_random = psf_from_phase(np.zeros((ADV_NPIX, ADV_NPIX)), sim_random["mask"])
_, ee_truth_random = encircled_energy(sim_random["psf_truth"], radii=[2.0, 5.0])
_, ee_recovered_random = encircled_energy(psf_recovered_random, radii=[2.0, 5.0])

df_psf_metrics_random = pd.DataFrame({
    "Métrique": ["Strehl", "FWHM H [px]", "FWHM V [px]", "EE(r=2px)", "EE(r=5px)"],
    "Simulée (vérité)": [
        strehl_ratio(sim_random["psf_truth"], psf_ideal_random),
        fwhm(x_px_random, sim_random["psf_truth"][cy_random, :]),
        fwhm(x_px_random, sim_random["psf_truth"][:, cx_random]),
        ee_truth_random[0], ee_truth_random[1],
    ],
    "Récupérée": [
        strehl_ratio(psf_recovered_random, psf_ideal_random),
        fwhm(x_px_random, psf_recovered_random[cy_random, :]),
        fwhm(x_px_random, psf_recovered_random[:, cx_random]),
        ee_recovered_random[0], ee_recovered_random[1],
    ],
})
print(df_psf_metrics_random.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

RMS WFE simulée (vérification) : 79.11 nm (budget demandé : 80.0 nm)
RMSE carte de WFE récupérée vs vérité : 0.00 nm


fc (fraction de fe) : 1.00


              Coupe  FTM(fe/4)  FTM(fe/2)  Intégrale [0, fe/2]
  Simulée — coupe H      0.471      0.323                0.269
  Simulée — coupe V      0.333      0.286                0.229
Récupérée — coupe H      0.471      0.323                0.269
Récupérée — coupe V      0.333      0.286                0.229
   Métrique  Simulée (vérité)  Récupérée
     Strehl             0.443      0.443
FWHM H [px]             1.151      1.151
FWHM V [px]             1.238      1.238
  EE(r=2px)             0.713      0.713
  EE(r=5px)             0.928      0.928


### Masque "calculé par ailleurs" : `aperture_px` auto-détecté

Si l'ouverture de `mask` ne remplit **pas** tout le cadre `npix` et que ce
masque vient d'ailleurs que de `PupilGenerator` (calculé par une autre
fonction, chargé puis recadré à la main...), `aperture_px` est
**auto-détecté** depuis l'étendue des pixels non nuls du masque
(`pupil.detect_aperture_px`, §12.6 de `REFERENCE_TECHNIQUE.md`) plutôt que
de supposer à tort qu'elle remplit `npix` entier -- l'erreur
historique illustrée ci-dessous, dont la gravité dépendait de `npix`.

In [21]:
from eoqual.phaseretrieval.pupil import detect_aperture_px

FIXED_APERTURE_PX = 64  # ouverture physique de taille fixe, indépendante de npix

print(f"{'npix':>6} {'Z4 vérité':>10} {'sans aperture_px (avant correctif)':>36} {'auto-détecté (après correctif)':>32}")
for npix_custom in (96, 128, 160, 256):
    # "Masque calculé par ailleurs" : généré indépendamment de npix, avec
    # une ouverture physique fixe -- la fraction aperture_px/npix change
    # donc avec npix, exactement le cas signalé.
    mask_custom = PupilGenerator(npix_custom, aperture_px=FIXED_APERTURE_PX).simple_disk()

    sim_custom = simulate_psf_from_zernike(
        truth_coeffs_random, wavelength=COMMON["wavelength"], npix=npix_custom,
        zernike_order=RANDOM_ORDER, mask=mask_custom,
    )
    opt_cfg_custom = OpticalConfig(wavelength=COMMON["wavelength"], zernike_order=RANDOM_ORDER, verbose=False)
    coeffs_custom, _ = fit_zernike_from_psf(sim_custom["psf"], mask_custom, opt_cfg_custom)
    coeffs_custom = resolve_sign_ambiguity(
        coeffs_custom, reference_index=4, expected_sign=np.sign(sim_custom["coeffs_nm"][3]) or 1.0,
    )

    # Pour comparaison : ce qu'aurait donné le comportement historique
    # (sans auto-détection), en forçant aperture_px=npix explicitement.
    basis_full_frame = resolve_zernike_basis(npix=npix_custom, nterms=RANDOM_ORDER, mask=mask_custom, aperture_px=npix_custom)
    phase_full_frame = phase_from_coefficients(
        sim_custom["coeffs_nm"] / rad_to_nm_random, basis_full_frame, mask_custom,
    )
    psf_full_frame = psf_from_phase(phase_full_frame, mask_custom)
    coeffs_bug, _ = fit_zernike_from_psf(psf_full_frame, mask_custom, OpticalConfig(
        wavelength=COMMON["wavelength"], zernike_order=RANDOM_ORDER, verbose=False, aperture_px=npix_custom,
    ))
    coeffs_bug = resolve_sign_ambiguity(coeffs_bug, 4, np.sign(sim_custom["coeffs_nm"][3]) or 1.0)

    print(f"{npix_custom:6d} {sim_custom['coeffs_nm'][3]:10.2f} {coeffs_bug[3]:36.2f} {coeffs_custom[3]:32.2f}")

print(f"\n(ouverture détectée : {detect_aperture_px(mask_custom)} px, pour une ouverture physique de {FIXED_APERTURE_PX} px)")

  npix  Z4 vérité   sans aperture_px (avant correctif)   auto-détecté (après correctif)


    96       7.26                                 2.46                             7.26


   128       7.26                                 1.41                             7.26


   160       7.26                                 0.28                             7.26


   256       7.26                                 9.19                             7.26

(ouverture détectée : 65 px, pour une ouverture physique de 64 px)


### Masque à résolution native différente de la PSF : `build_zernike_basis` direct a besoin de `aperture_px=n_pup_target`

Quand le masque est chargé depuis un fichier à sa **propre résolution
native** (mesurée ou calculée ailleurs, sans rapport avec le cadre
`npix` de la PSF) via `load_and_prepare_mask`, celui-ci renvoie
`(mask, N_pup_target)` — `N_pup_target` est le nombre de pixels que
l'ouverture occupe réellement dans le cadre `npix`, quasiment toujours
**plus petit** que `npix` (c'est la taille physiquement correcte,
cf. §12 et `compute_pupil_footprint_px`).

Si la phase de **vérité** (pour un test, une simulation...) est ensuite
construite en appelant `build_zernike_basis` **directement** — pas via
`resolve_zernike_basis` — `aperture_px` doit être passé explicitement,
avec cette valeur `N_pup_target`. `build_zernike_basis` ne prend pas de
`mask` en argument : contrairement à `resolve_zernike_basis` (§12.6,
auto-détection depuis un masque), il n'a **rien** depuis quoi
auto-détecter — sans `aperture_px`, il suppose silencieusement que
l'ouverture remplit tout le cadre `npix`.

C'est exactement le piège illustré ci-dessous : l'ajustement
(`fit_zernike_from_psf`/`psf_to_wfe`), lui, passe par
`resolve_zernike_basis` en interne et détecte donc correctement
l'ouverture depuis le masque — seule la **vérité**, construite à la
main avec `build_zernike_basis`, reste mal échelonnée si `aperture_px`
est oublié. Résultat : un coût d'optimisation très faible (l'ajustement
reproduit très bien *son propre* modèle, correctement échelonné) mais
des coefficients de Zernike — et une RMSE de WFE — qui n'ont rien à
voir avec la vérité injectée.


In [22]:
import tempfile

import imageio.v3 as iio

from eoqual.phaseretrieval.io_utils import load_and_prepare_mask

# Masque "mesuré/calculé ailleurs" à sa propre résolution native (ici
# 1024x1024 px), sans rapport avec le cadre npix de la PSF -- exactement
# le cas d'un fichier masque produit par un autre outil.
native_mask = PupilGenerator(1024).simple_disk()
native_mask_path = Path(tempfile.mkdtemp()) / "mask_native.tiff"
iio.imwrite(native_mask_path, (native_mask * 255).astype(np.uint8))

NATIVE_PARAMS = dict(wavelength=650e-9, focal_length=30.0, pupil_diameter=1.5, pixel_pitch=10e-6)
NPIX_NATIVE = 128
ORDER_NATIVE = 15

# load_and_prepare_mask recadre/rééchantillonne le masque natif sur le
# cadre de la PSF et renvoie N_pup_target : la taille (en pixels) que
# l'ouverture occupe réellement dans ce cadre -- 98 px sur 128, pas 128.
mask_native, n_pup_target_native = load_and_prepare_mask(
    psf_shape=(NPIX_NATIVE, NPIX_NATIVE), mask_path=native_mask_path, **NATIVE_PARAMS,
)
mask_native = mask_native.astype(bool)
print(f"Ouverture native rééchantillonnée : N_pup_target = {n_pup_target_native} px (cadre {NPIX_NATIVE} px)")

rad_to_nm_native = NATIVE_PARAMS["wavelength"] * 1e9 / (2 * np.pi)
coeffs_nm_native = np.zeros(ORDER_NATIVE)
coeffs_nm_native[3:] = [-15.0, 13.0, 13.0, -11.0, 9.0, 7.0, -6.0, -3.0, -3.0, 1.0, 1.0, 0.0]
opt_cfg_native = OpticalConfig(wavelength=NATIVE_PARAMS["wavelength"], zernike_order=ORDER_NATIVE, verbose=False)


def truth_and_fit(aperture_px):
    """Vérité construite avec ``build_zernike_basis(..., aperture_px=aperture_px)``, puis ajustement."""
    basis = build_zernike_basis(npix=NPIX_NATIVE, nterms=ORDER_NATIVE, aperture_px=aperture_px)
    phase = phase_from_coefficients(coeffs_nm_native / rad_to_nm_native, basis, mask_native)
    psf = psf_from_phase(phase, mask_native)
    coeffs, diag = fit_zernike_from_psf(psf, mask_native, opt_cfg_native)
    coeffs = resolve_sign_ambiguity(coeffs, reference_index=4, expected_sign=np.sign(coeffs_nm_native[3]))
    return coeffs, diag


# "Avant correctif" : aperture_px omis -- build_zernike_basis suppose à
# tort que l'ouverture remplit tout le cadre (128 px) alors qu'elle n'en
# occupe que 98 (n_pup_target_native, ignoré ici par erreur).
coeffs_bug, diag_bug = truth_and_fit(aperture_px=None)

# "Après correctif" : aperture_px=n_pup_target_native -- même échelle
# d'ouverture pour la vérité et pour l'ajustement (qui, lui, la détecte
# déjà seul depuis mask_native via resolve_zernike_basis, §12.6).
coeffs_fixed, diag_fixed = truth_and_fit(aperture_px=n_pup_target_native)

print(f"\n{'Zernike':>8} {'vérité':>10} {'sans aperture_px':>18} {'aperture_px=n_pup_target':>26}")
for i in range(3, ORDER_NATIVE):
    print(f"{'Z' + str(i + 1):>8} {coeffs_nm_native[i]:10.2f} {coeffs_bug[i]:18.2f} {coeffs_fixed[i]:26.2f}")

print(f"\nCoût d'optimisation -- sans aperture_px : {diag_bug['cost']:.2e} | aperture_px=n_pup_target : {diag_fixed['cost']:.2e}")
print("(coût faible dans les deux cas : l'ajustement réussit bien sur SON modèle -- seule la vérité, mal échelonnée sans aperture_px, ne correspond pas aux coefficients injectés)")


2026-11-01 15:21:07.562 | INFO     | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:487 - Ouverture estimée à 98x98 px sur la grille de la PSF (128x128) — masque natif 1024x1024 px.


2026-11-01 15:21:07.586 | DEBUG    | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:508 - Mask sum (number of illuminated pixels): 7866.0


2026-11-01 15:21:07.586 | DEBUG    | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:509 - Mask shape:   (128, 128)


2026-11-01 15:21:07.586 | DEBUG    | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:510 - N_pup_target: 98


Ouverture native rééchantillonnée : N_pup_target = 98 px (cadre 128 px)



 Zernike     vérité   sans aperture_px   aperture_px=n_pup_target
      Z4     -15.00              -9.16                     -15.00
      Z5      13.00               6.19                      13.00
      Z6      13.00              11.17                      13.00
      Z7     -11.00              -5.17                     -11.00
      Z8       9.00               4.18                       9.00
      Z9       7.00               3.65                       7.00
     Z10      -6.00              -2.81                      -6.00
     Z11      -3.00               1.94                      -3.00
     Z12      -3.00              -0.23                      -3.00
     Z13       1.00              -2.53                       1.00
     Z14       1.00              -3.48                       1.00
     Z15       0.00              -3.54                      -0.00

Coût d'optimisation -- sans aperture_px : 5.64e-03 | aperture_px=n_pup_target : 1.58e-17
(coût faible dans les deux cas : l'ajustement réuss

### Vérification visuelle : le recadrage (`_crop_to_aperture`) a bien lieu *avant* le changement d'échelle

Le masque natif de l'exemple précédent remplissait tout son cadre
(1024×1024 px, aucune marge) — un cas un peu particulier qui ne testait
pas le recadrage. Un fichier masque réel (comme `mask.tif`) a presque
toujours une marge de zéros autour de la pupille. Reprenons avec une
ouverture qui n'occupe qu'une partie du fichier natif, et affichons le
résultat pour vérifier que `load_and_prepare_mask` recadre bien l'ouverture
**avant** de la redimensionner — pas l'inverse, ce qui donnerait une
ouverture trop petite (la marge serait comprimée avec elle).


In [23]:
import tempfile

import imageio.v3 as iio
from plotly.subplots import make_subplots
from skimage.transform import resize

from eoqual.phaseretrieval.io_utils import load_and_prepare_mask
from eoqual.phaseretrieval.pupil import detect_aperture_px

# Masque natif réaliste : l'ouverture n'occupe qu'une partie du fichier,
# avec une vraie marge de zéros tout autour (contrairement au disque
# "plein cadre" de l'exemple précédent) -- le cas d'un vrai mask.tif.
NATIVE_FRAME_PX = 1024
NATIVE_APERTURE_PX = 700  # ouverture dans le fichier natif ; marge ~162 px de chaque côté
native_mask_margin = PupilGenerator(NATIVE_FRAME_PX, aperture_px=NATIVE_APERTURE_PX).simple_disk()
native_mask_margin_path = Path(tempfile.mkdtemp()) / "mask_native_margin.tiff"
iio.imwrite(native_mask_margin_path, (native_mask_margin * 255).astype(np.uint8))

mask_margin_reconstructed, n_pup_target_margin = load_and_prepare_mask(
    psf_shape=(NPIX_NATIVE, NPIX_NATIVE), mask_path=native_mask_margin_path, **NATIVE_PARAMS,
)
mask_margin_reconstructed = mask_margin_reconstructed.astype(bool)

# Pour comparaison : un resize NAIF du fichier natif ENTIER (marge
# comprise) directement vers n_pup_target_margin, sans passer par
# _crop_to_aperture au préalable -- reproduit "à la main" ce que donnerait
# l'ordre inverse (resize puis crop, ou pas de crop du tout).
naive_resized = resize(
    native_mask_margin, output_shape=(n_pup_target_margin, n_pup_target_margin),
    order=1, anti_aliasing=True, preserve_range=True,
)
naive_canvas = np.zeros((NPIX_NATIVE, NPIX_NATIVE))
off_naive = (NPIX_NATIVE - n_pup_target_margin) // 2
naive_canvas[off_naive:off_naive + n_pup_target_margin, off_naive:off_naive + n_pup_target_margin] = naive_resized
naive_canvas = naive_canvas > 0.5

aperture_correct = detect_aperture_px(mask_margin_reconstructed)
aperture_naive = detect_aperture_px(naive_canvas)
print(f"Ouverture native : {NATIVE_APERTURE_PX} px dans un fichier {NATIVE_FRAME_PX}x{NATIVE_FRAME_PX} -> N_pup_target = {n_pup_target_margin} px (cadre {NPIX_NATIVE} px)")
print(f"load_and_prepare_mask (crop PUIS resize)      : ouverture mesurée = {aperture_correct} px (attendu {n_pup_target_margin} px -- correct)")
print(f"resize naïf du fichier entier (sans crop avant) : ouverture mesurée = {aperture_naive} px (marge comprimée avec l'ouverture -- FAUX)")

fig_crop_check = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        f"Masque natif ({NATIVE_FRAME_PX}x{NATIVE_FRAME_PX}, ouverture {NATIVE_APERTURE_PX} px, marge visible)",
        f"load_and_prepare_mask : crop puis resize<br>(ouverture = {aperture_correct} px, correct)",
        f"Resize naïf sans crop préalable<br>(ouverture = {aperture_naive} px, FAUX)",
    ),
    horizontal_spacing=0.08,
)
fig_crop_check.add_heatmap(z=native_mask_margin, colorscale="Greys", showscale=False, row=1, col=1)
fig_crop_check.add_heatmap(z=mask_margin_reconstructed.astype(float), colorscale="Greys", showscale=False, row=1, col=2)
fig_crop_check.add_heatmap(z=naive_canvas.astype(float), colorscale="Greys", showscale=False, row=1, col=3)
for col in (1, 2, 3):
    fig_crop_check.update_yaxes(scaleanchor=f"x{col if col > 1 else ''}", scaleratio=1, row=1, col=col)
fig_crop_check.update_layout(
    height=380, width=980,
    title="Masque natif (avec marge) vs reconstruction correcte vs resize naïf",
    margin=dict(l=0, r=0, t=80, b=0),
)
fig_crop_check.show()


2026-11-01 15:21:07.878 | INFO     | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:487 - Ouverture estimée à 98x98 px sur la grille de la PSF (128x128) — masque natif 1024x1024 px.


2026-11-01 15:21:07.887 | DEBUG    | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:508 - Mask sum (number of illuminated pixels): 7836.0


2026-11-01 15:21:07.887 | DEBUG    | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:509 - Mask shape:   (128, 128)


2026-11-01 15:21:07.887 | DEBUG    | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:510 - N_pup_target: 98


Ouverture native : 700 px dans un fichier 1024x1024 -> N_pup_target = 98 px (cadre 128 px)
load_and_prepare_mask (crop PUIS resize)      : ouverture mesurée = 98 px (attendu 98 px -- correct)
resize naïf du fichier entier (sans crop avant) : ouverture mesurée = 67 px (marge comprimée avec l'ouverture -- FAUX)
